<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/04_nlp/01_text_classification_quest_afrisenti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💬 Quest · Text classification: How does Naija feel today?

**"Wahala dey o!"** On Nigerian Twitter, people complain about fuel prices, celebrate football wins, and roast each
other, often in **Nigerian Pidgin** (*Naija*), spoken by more than 100 million people. Can a machine read the mood
of these tweets?

**A little history.** In 1966, Joseph Weizenbaum's *ELIZA* fooled people into thinking it understood them, with
nothing but keyword rules. In 2002, Paul Graham's essay *A Plan for Spam* showed that counting words and applying
Bayes' rule could kill most spam, and the same year Pang & Lee taught computers to tell positive from negative movie
reviews. For decades almost all of this work was done in English. In 2019 a grassroots community, **Masakhane**
("we build together" in isiZulu), set out to change that for African languages. One of its results is **AfriSenti**
(SemEval-2023 shared task), 110 000+ tweets in 14 African languages labelled *positive*, *neutral* or *negative*.

**Your mission:** predict the sentiment of Nigerian Pidgin tweets.
**Metric:** **macro-F1** (the average F1 of the 3 classes, so the rare *neutral* class counts as much as the others).

| You will practise | Concept |
|---|---|
| Turning text into tokens | tokenisation, normalisation |
| Turning tokens into numbers | bag-of-words, TF-IDF, n-grams |
| A strong classical baseline | logistic regression on sparse features |
| Choosing the right metric | accuracy vs macro-F1 on imbalanced classes |
| Understanding errors | confusion matrix, top words, failure cases |

🧩 Cells marked `# TODO` are yours. Replace the `...`, then run the `check(...)` cell below.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

quest("How does Naija feel today?", chapter="Track 4 · NLP · Text classification", xp=300)

## 1. Load the tweets 📥

AfriSenti is published on the Hugging Face Hub. `hf_parquet` downloads the official Parquet export into `data/hf/`
(no account needed). The dataset comes with three splits: `train` to learn, `validation` to compare ideas, `test` to
measure once at the very end.

In [ ]:
LANG = "pcm"     # Nigerian Pidgin. Other options: "swa" (Swahili), "hau" (Hausa), "yor" (Yoruba), "ibo" (Igbo), "amh"...
train = hf_parquet("masakhane/afrisenti", "train", LANG)
valid = hf_parquet("masakhane/afrisenti", "validation", LANG)
test = hf_parquet("masakhane/afrisenti", "test", LANG)
print(train.shape, valid.shape, test.shape)
pd.set_option("display.max_colwidth", 120)
train.sample(8, random_state=SEED)

### 🎯 Exercise 1: how imbalanced are the classes?
Compute the share of each label in `train` (a Series that sums to 1), and the accuracy of a model that always
predicts the most frequent class.

In [ ]:
label_share = ...          # TODO: hint: value_counts(normalize=True)
majority_accuracy = ...    # TODO: the largest share
print(label_share, f"\nAlways predicting the majority class gives {majority_accuracy:.1%} accuracy")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
label_share = train["label"].value_counts(normalize=True)
majority_accuracy = label_share.max()
print(label_share, f"\nAlways predicting the majority class gives {majority_accuracy:.1%} accuracy")
```

</details>

In [ ]:
check(abs(label_share.sum() - 1) < 1e-6 and 0.55 < majority_accuracy < 0.7,
      "Right: a lazy model scores ~63% accuracy without reading a single word, and neutral tweets are only ~1.4%!",
      "label_share should be train['label'].value_counts(normalize=True); majority_accuracy its max.")

> 🧠 **Why macro-F1?** F1 combines *precision* (when I say "neutral", am I right?) and *recall* (did I find all the
> neutral tweets?). Macro-F1 averages it over the classes **without weighting by size**, so a model that ignores
> the rare class gets punished. The lazy model above has a macro-F1 of about 0.26.
>
> In Pidgin AfriSenti, *neutral* is extremely rare (~70 tweets out of 5 121). Keep an eye on it: it will be the
> hardest class for every model in this notebook.

## 2. From text to tokens ✂️

A computer does not see words, it sees characters. **Tokenisation** cuts a text into units (*tokens*). The simplest
recipe: lowercase everything and keep sequences of letters/digits.

### 🎯 Exercise 2: write a tokenizer

In [ ]:
def tokenize(text: str) -> list[str]:
    # TODO: lowercase the text, then return all the "words" (hint: re.findall(r"\w+", ...))
    ...

tokenize("Wahala DEY o!! Fuel don cost 617 naira 😭")

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

tokenize("Wahala DEY o!! Fuel don cost 617 naira 😭")
```

</details>

In [ ]:
check(tokenize("Wahala DEY o!! Fuel don cost 617 naira 😭") == ["wahala", "dey", "o", "fuel", "don", "cost", "617", "naira"],
      "Clean tokens! Notice the emoji disappeared: is that a good idea for sentiment? (Keep that question in mind.)",
      "Expected ['wahala', 'dey', 'o', 'fuel', 'don', 'cost', '617', 'naira'].")

In [ ]:
from collections import Counter
counts = Counter(tok for t in train["tweet"] for tok in tokenize(t))
print(f"{len(counts):,} different tokens in {len(train):,} tweets. The 25 most common:")
print(counts.most_common(25))

The most common tokens (*i*, *the*, *dey*, *na*, *no*…) are **stop words**: frequent everywhere, informative nowhere.
Mixed English/Pidgin text is also typical of social media: this is called **code-switching**.

## 3. Bag-of-words: a text becomes a vector 🎒

The **bag-of-words** idea (1950s information retrieval) is brutal and effective: forget word order, count how many
times each vocabulary word appears. Each tweet becomes a long vector with mostly zeros (*sparse*).

```
"i dey happy, happy die"  →  {dey: 1, happy: 2, i: 1, die: 1}  →  [0, 0, 1, 0, ..., 2, ..., 1, 0]
```

`CountVectorizer` does tokenisation + counting in one go.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.dummy import DummyClassifier

lb = Leaderboard("afrisenti_pcm", metric="macro_F1")

def evaluate(name, model):
    """Fit on train, score macro-F1 on validation, log it."""
    model.fit(train["tweet"], train["label"])
    score = f1_score(valid["label"], model.predict(valid["tweet"]), average="macro")
    lb.add(name, score)

evaluate("Dummy (majority class)", DummyClassifier(strategy="most_frequent"))

### 🎯 Exercise 3: bag-of-words + logistic regression
Build a pipeline: `CountVectorizer(min_df=2)` (ignore words seen only once) then `LogisticRegression(max_iter=2000)`.

In [ ]:
bow_model = ...    # TODO: make_pipeline(..., ...)
evaluate("Bag-of-words + LogReg", bow_model)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
bow_model = make_pipeline(CountVectorizer(min_df=2), LogisticRegression(max_iter=2000))
evaluate("Bag-of-words + LogReg", bow_model)
```

</details>

In [ ]:
check(lb.best()["macro_F1"] > 0.40,
      "Huge jump over the dummy: counting words already captures a lot of sentiment.",
      "Did you build make_pipeline(CountVectorizer(min_df=2), LogisticRegression(max_iter=2000))?")
checkpoint("afrisenti/bow", xp=60)

## 4. TF-IDF and n-grams: smarter counting 🧮

Two upgrades that won many competitions before deep learning:

- **TF-IDF** (*term frequency × inverse document frequency*, Karen Spärck Jones, 1972): a word that appears in every
  tweet ("the", "i") gets a small weight; a rarer, more specific word ("wahala", "congrats") gets a large one.
- **n-grams**: also count pairs of consecutive words (*bigrams*): "not good" ≠ "good".
- **character n-grams**: count chunks of 2-5 letters. Perfect for Pidgin, where the same word is spelled many ways
  (*dey/dei/de*, *abeg/abeeg*), and for typos, hashtags and elongations (*soooo*).

### 🎯 Exercise 4: combine word and character TF-IDF
`make_union` concatenates the features of several vectorizers side by side.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union

features = make_union(
    TfidfVectorizer(...),     # TODO: word unigrams + bigrams: ngram_range=(1, 2), min_df=2, sublinear_tf=True
    TfidfVectorizer(...),     # TODO: characters: analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True
)
tfidf_model = make_pipeline(features, LogisticRegression(max_iter=3000, C=3, class_weight="balanced"))
evaluate("TF-IDF words+chars + LogReg", tfidf_model)
lb.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union

features = make_union(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
)
tfidf_model = make_pipeline(features, LogisticRegression(max_iter=3000, C=3, class_weight="balanced"))
evaluate("TF-IDF words+chars + LogReg", tfidf_model)
lb.show()
```

</details>

In [ ]:
check(lb.best()["model"].startswith("TF-IDF"),
      "TF-IDF + character n-grams win. `class_weight='balanced'` is meant to help the rare neutral class: does it? See the report below.",
      "The TF-IDF model should beat the bag-of-words one. Check the two vectorizers' parameters.")
checkpoint("afrisenti/tfidf", xp=60)

## 5. 🔬 Error analysis: what did the model learn, where does it fail?

A score is a summary. To improve, you have to **look** at the errors.

In [ ]:
pred = tfidf_model.predict(valid["tweet"])
print(classification_report(valid["label"], pred, digits=3))
ConfusionMatrixDisplay.from_predictions(valid["label"], pred, cmap="Blues", normalize="true", values_format=".0%")
plt.title("Confusion matrix (rows = truth)"); plt.grid(False); plt.show()

### 🎯 Exercise 5: the most "positive" and "negative" words
A logistic regression has one coefficient per feature and class: large positive coefficient = strong evidence for that
class. Fit the simple **word-level** TF-IDF model below and display its top-15 words for `positive` and `negative`.

In [ ]:
word_model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, C=3))
word_model.fit(train["tweet"], train["label"])
vocab = word_model[0].get_feature_names_out()
clf = word_model[-1]
top_words = {}
for label in ["positive", "negative"]:
    k = list(clf.classes_).index(label)
    top_words[label] = ...     # TODO: the 15 words with the largest coefficients clf.coef_[k] (hint: np.argsort)
pd.DataFrame(top_words)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
word_model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True), LogisticRegression(max_iter=2000, C=3))
word_model.fit(train["tweet"], train["label"])
vocab = word_model[0].get_feature_names_out()
clf = word_model[-1]
top_words = {}
for label in ["positive", "negative"]:
    k = list(clf.classes_).index(label)
    top_words[label] = vocab[np.argsort(clf.coef_[k])[::-1][:15]]
pd.DataFrame(top_words)
```

</details>

In [ ]:
check(len(top_words.get("positive", [])) == 15 and len(top_words.get("negative", [])) == 15,
      "Read the two columns: do they make sense to a Naija speaker? Spotting nonsense words is how you find data issues.",
      "top_words[label] should contain 15 words, picked with np.argsort(clf.coef_[k])[::-1][:15].")

In [ ]:
errors = valid.assign(pred=pred).query("label != pred").sample(10, random_state=SEED)
errors[["tweet", "label", "pred"]]

Read these errors like a detective. In sentiment analysis the usual suspects are **sarcasm**, **mixed feelings**,
tweets whose sentiment lives in an **emoji** we threw away, **missing context** (a reply to a tweet we can't see),
and labels you would disagree with yourself (annotators disagree too: no model can beat label noise).

And look again at the classification report: the model almost never predicts *neutral*. With ~70 examples, there is
simply not enough signal. In a competition, this is where you would hunt for more data or tune the decision
thresholds per class.

## 6. 🏁 Final evaluation on the test set

Validation was used to choose the model. Now we refit on **train + validation** and evaluate **once** on the test set.

In [ ]:
full = pd.concat([train, valid])
tfidf_model.fit(full["tweet"], full["label"])
test_pred = tfidf_model.predict(test["tweet"])
test_f1 = f1_score(test["label"], test_pred, average="macro")
print(f"🏁 Test macro-F1: {test_f1:.3f}")
make_submission(pd.DataFrame({"id": test.index, "label": test_pred}), f"afrisenti_{LANG}_submission.csv");

## 7. 🎤 Talk to your model

In [ ]:
my_tweets = [
    "Abeg this NEPA don take light again, I tire for this country",
    "Super Eagles win! Na celebration everywhere tonight 🎉",
    "I go market tomorrow morning",
    "Omo this jollof sweet die, God bless the cook",
]
for t, p, proba in zip(my_tweets, tfidf_model.predict(my_tweets), tfidf_model.predict_proba(my_tweets).max(1)):
    print(f"{p:>8} ({proba:.0%})  ←  {t}")

Not every answer is right. Look at the probabilities: around 50-60% the model is hesitating, often because the
words it relies on ("win", "celebration") were rare in its 5 000 training tweets. More data, or a model pre-trained
on millions of sentences, fixes much of this.

## 🏆 Quest complete!

You built the classic NLP pipeline: tokenise → vectorise → linear model → error analysis. Don't underestimate it:
TF-IDF + logistic regression is fast, explainable, and a baseline that transformers must beat (they do, by a few
points, see the next notebook).

### 🚀 Your turn (bonus XP)
1. Keep emojis: use `TfidfVectorizer(token_pattern=r"[^\s]+")` or add an emoji count feature. Does neutral improve?
2. Switch `LANG` to `"hau"` (Hausa: 14 000 tweets, balanced classes) or `"yor"`, `"ibo"`, `"swa"`. Compare the
   macro-F1 across languages. Which is hardest, and why?
3. Try `LinearSVC` or `ComplementNB` (naive Bayes, the spam-filter classic) instead of the logistic regression.
4. Go further: `02_text_classification_app_swahili_news.ipynb` fine-tunes an African-language transformer.

In [ ]:
checkpoint("afrisenti/complete", xp=180)
badge("Naija Mood Reader", "💬")
progress()